# 07 — Multi-Agent: Swarm

**Stage 7 of the workshop.** Self-organizing agents that hand off to each other with no fixed routing — each agent decides on its own whether to hand off, and to whom.

## Problem

One agent, one system prompt, one toolset gets unwieldy fast. Workflow and Graph both require *you* to wire the routing (in code, or via conditions). Swarm removes even that — agents decide their own handoffs.

## Concept

Unlike Graph (explicit edges + conditions you write), a Swarm has no fixed routing: each agent decides on its own whether to hand off, and to whom, by calling a `handoff_to_agent` tool. Loop guards (`max_handoffs`, `execution_timeout`, repetitive-handoff detection) are what keep that freedom from turning into an infinite bounce — this is the natural extreme of the Workflow → Graph → Swarm control spectrum: least control, most emergent behavior.

## Architecture

```
research_agent ──handoff_to_agent("creative_agent")──▶ creative_agent
                                                            │
                                       handoff_to_agent("critical_agent")
                                                            ▼
                                                     critical_agent
                                                            │
                                    handoff_to_agent("summarizer_agent")
                                                            ▼
                                                   summarizer_agent (no handoff — final answer)
```

Each agent's system prompt explicitly instructs it which agent to hand off to next (and to actually invoke the tool, not just narrate it) — the swarm mechanism itself imposes no fixed order, the prompts create this particular chain.

## Step 1 — Model and imports

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model
from strands import Agent
from strands.multiagent import Swarm

model = get_model()


## Step 2 — Define the four agents

Each one's system prompt tells it what to do and which agent (if any) to hand off to.

In [ ]:
research_agent = Agent(
    model=model,
    name="research_agent",
    system_prompt=(
        "You are a Research Agent. Give 2-3 short factual bullet points on "
        "the topic. Then you MUST call the handoff_to_agent tool with "
        "agent_name=\"creative_agent\" — do not answer in prose only, "
        "actually invoke the tool. Never hand off to yourself."
    ),
)

creative_agent = Agent(
    model=model,
    name="creative_agent",
    system_prompt=(
        "You are a Creative Agent. Read the research so far and propose "
        "one clear, practical angle in 2-3 sentences. Then you MUST call "
        "the handoff_to_agent tool with agent_name=\"critical_agent\" — "
        "do not answer in prose only, actually invoke the tool."
    ),
)

critical_agent = Agent(
    model=model,
    name="critical_agent",
    system_prompt=(
        "You are a Critical Agent. Point out one real weakness in the "
        "proposal so far, in 1-2 sentences. Then you MUST call the "
        "handoff_to_agent tool with agent_name=\"summarizer_agent\" — do "
        "not answer in prose only, actually invoke the tool."
    ),
)

summarizer_agent = Agent(
    model=model,
    name="summarizer_agent",
    system_prompt=(
        "You are a Summarizer Agent. Combine the research, the proposal, "
        "and the critique into one short final answer (<=80 words). Do "
        "not call handoff_to_agent — this is the last step, just answer."
    ),
)


## Step 3 — Build the swarm with loop guards

`max_handoffs`, `max_iterations`, `execution_timeout`, `node_timeout`, and repetitive-handoff detection all bound the otherwise-unfixed routing.

In [ ]:
swarm = Swarm(
    [research_agent, creative_agent, critical_agent, summarizer_agent],
    max_handoffs=8,
    max_iterations=8,
    execution_timeout=420.0,
    node_timeout=150.0,
    repetitive_handoff_detection_window=6,
    repetitive_handoff_min_unique_agents=3,
)


## Step 4 — Run it

In [ ]:
result = swarm("Explain Agentic AI in one short blog-post-style summary.")
print(f"Status: {result.status}")
for node in result.node_history:
    print(f"Agent: {node.node_id}")
print(result)


## Failure mode to know about

Without loop guards, a swarm's freedom to hand off however agents choose can turn into an infinite bounce between agents — `max_handoffs`, `execution_timeout`, and repetitive-handoff detection are what keep that from happening, the same principle as Graph's `max_node_executions`/`execution_timeout` but applied to routing the agents choose themselves rather than routing you wrote.